### Import libraries

In [1]:
from pathlib import Path
import ast
import joblib
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import save_npz

### Set paths

In [2]:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = ROOT / "data" / "processed"
FEATURES_DIR = ROOT / "data" / "interim" / "tfidf_features"
MODEL_DIR = ROOT / "models" / "traditional_ml"

FEATURES_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

### Load datasets

In [3]:
train_df = pd.read_csv(DATA_DIR / "train_tokenized.csv")
validation_df = pd.read_csv(DATA_DIR / "validation.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

### Convert tokens into text

In [4]:
def tokens_to_text(value):
    if pd.isna(value):
        return ""

    if isinstance(value, list):
        return " ".join(map(str, value))

    value = str(value).strip()

    if value.startswith("[") and value.endswith("]"):
        try:
            parsed = ast.literal_eval(value)

            if isinstance(parsed, list):
                return " ".join(map(str, parsed))
        except (ValueError, SyntaxError):
            pass

    return value


for split_name, df in [
    ("train", train_df),
    ("validation", validation_df),
    ("test", test_df),
]:
    if "sentiment" not in df.columns:
        raise ValueError(f"Missing sentiment column in {split_name} dataset")

    text_column = "tokens" if "tokens" in df.columns else "clean_text"
    if text_column not in df.columns:
        raise ValueError(
            f"Missing both 'tokens' and 'clean_text' columns in {split_name} dataset"
        )

    df["tfidf_text"] = df[text_column].apply(tokens_to_text)
    df["sentiment"] = df["sentiment"].astype(str).str.strip().str.lower()

    df.drop(
        df[
            (df["tfidf_text"].str.strip() == "") |
            (df["sentiment"] == "") |
            (df["sentiment"] == "nan")
        ].index,
        inplace=True
    )

    df.reset_index(drop=True, inplace=True)

print("Tokenized text is ready.")

Tokenized text is ready.


### Create TF-IDF vectorizer

In [5]:
tfidf = TfidfVectorizer(
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    lowercase=False,
    ngram_range=(1, 2),
    max_features=50000,
    min_df=2,
    sublinear_tf=True
)

### Fit on training data

In [6]:
X_train = tfidf.fit_transform(train_df["tfidf_text"])
y_train = train_df["sentiment"]

print("Training features:", X_train.shape)

Training features: (15991, 50000)


### Transform validation and test data

In [7]:
X_validation = tfidf.transform(validation_df["tfidf_text"])
y_validation = validation_df["sentiment"]

X_test = tfidf.transform(test_df["tfidf_text"])
y_test = test_df["sentiment"]

print("Validation features:", X_validation.shape)
print("Test features:", X_test.shape)

Validation features: (1999, 50000)
Test features: (1999, 50000)


### Save the vectorizer

In [8]:
joblib.dump(tfidf, MODEL_DIR / "tfidf_vectorizer.joblib")

print("TF-IDF vectorizer saved.")

TF-IDF vectorizer saved.


### Save feature matrices

In [9]:
save_npz(FEATURES_DIR / "X_train_tfidf.npz", X_train)
save_npz(FEATURES_DIR / "X_validation_tfidf.npz", X_validation)
save_npz(FEATURES_DIR / "X_test_tfidf.npz", X_test)

print("TF-IDF matrices saved.")

TF-IDF matrices saved.


### Save labels

In [10]:
y_train.to_frame(name="sentiment").to_csv(
    FEATURES_DIR / "y_train.csv", index=False
)

y_validation.to_frame(name="sentiment").to_csv(
    FEATURES_DIR / "y_validation.csv", index=False
)

y_test.to_frame(name="sentiment").to_csv(
    FEATURES_DIR / "y_test.csv", index=False
)

print("Labels saved.")

Labels saved.
